# agent_10 — Financial Sentiment Analysis Agent

An agentic AI workflow for financial tweet sentiment classification.
The agent maintains a conversational interface and orchestrates two classification
models — a fast TF-IDF pipeline and a fine-tuned FinBERT — using OpenAI function
calling against a DeepSeek-compatible API endpoint.

## What this demonstrates

| Requirement | Implementation |
|---|---|
| Conversational interface | Message-history loop across multiple turns |
| Tool use | 4 OpenAI function-calling tools |
| Non-trivial routing | Agent selects model based on intent; auto-escalates on disagreement |

## Models available to the agent

| Model | Macro F1 (val) | Speed |
|---|---|---|
| TF-IDF + Logistic Regression | 0.741 | < 1 ms |
| Fine-tuned FinBERT (ProsusAI/finbert) | 0.822 | ~1 s |


In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FINBERT_MODEL_DIR = PROCESSED_DATA_DIR / 'finbert_final_model'

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from preprocessing import regex_clean, stem_text

print(f'Project root:          {PROJECT_ROOT}')
print(f'FinBERT model dir:     {FINBERT_MODEL_DIR}')
print(f'FinBERT model exists:  {FINBERT_MODEL_DIR.exists()}')


In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
X = train['text']
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=73
)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Label distribution: {dict(y_train.value_counts().sort_index())}')


In [ ]:
print('Applying preprocessing to training data...')
X_train_stemmed = X_train.apply(lambda t: stem_text(regex_clean(str(t).lower())))
X_val_stemmed   = X_val.apply(lambda t: stem_text(regex_clean(str(t).lower())))

print('Fitting TF-IDF + balanced Logistic Regression (fast model)...')
fast_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=73)),
])
fast_pipeline.fit(X_train_stemmed, y_train)

val_acc = fast_pipeline.score(X_val_stemmed, y_val)
print(f'Fast model ready. Validation accuracy: {val_acc:.4f}')


In [ ]:
LABEL_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else
    'cpu'
)

_finbert_model     = None
_finbert_tokenizer = None


def get_finbert():
    global _finbert_model, _finbert_tokenizer
    if _finbert_model is None:
        print('Loading fine-tuned FinBERT from disk (first call only)...')
        _finbert_tokenizer = AutoTokenizer.from_pretrained(str(FINBERT_MODEL_DIR))
        _finbert_model = (
            AutoModelForSequenceClassification
            .from_pretrained(str(FINBERT_MODEL_DIR))
            .to(DEVICE)
        )
        _finbert_model.eval()
        print(f'FinBERT loaded on {DEVICE}.')
    return _finbert_model, _finbert_tokenizer


print(f'Inference device: {DEVICE}')
print('FinBERT will be loaded on the first classify_best() or compare_models() call.')


In [ ]:
def classify_fast(tweet: str) -> dict:
    """Classify using TF-IDF + Logistic Regression (fast path)."""
    preprocessed = stem_text(regex_clean(str(tweet).lower()))
    label  = int(fast_pipeline.predict([preprocessed])[0])
    probas = fast_pipeline.predict_proba([preprocessed])[0]
    return {
        'label':                label,
        'sentiment':            LABEL_NAMES[label],
        'confidence':           {LABEL_NAMES[i]: round(float(p), 3) for i, p in enumerate(probas)},
        'model':                'TF-IDF + Logistic Regression',
        'macro_f1_validation':  0.741,
    }


def classify_best(tweet: str) -> dict:
    """Classify using fine-tuned FinBERT (most accurate)."""
    model, tokenizer = get_finbert()
    inputs = tokenizer(
        str(tweet),
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    label = int(logits.argmax(dim=-1).item())
    return {
        'label':                label,
        'sentiment':            LABEL_NAMES[label],
        'confidence':           {LABEL_NAMES[i]: round(float(p), 3) for i, p in enumerate(probs)},
        'model':                'Fine-tuned FinBERT (ProsusAI/finbert)',
        'macro_f1_validation':  0.822,
    }


def compare_models(tweet: str) -> dict:
    """Run both models and compare predictions."""
    fast = classify_fast(tweet)
    best = classify_best(tweet)
    agreement = fast['label'] == best['label']
    if agreement:
        recommendation = (
            f"Both models agree: {best['sentiment']}. "
            'Fine-tuned FinBERT result is the most reliable.'
        )
    else:
        recommendation = (
            f"Models disagree — fast model: {fast['sentiment']}, "
            f"best model: {best['sentiment']}. "
            'Defer to fine-tuned FinBERT (macro F1 0.822 vs 0.741).'
        )
    return {
        'fast_model':     fast,
        'best_model':     best,
        'agreement':      agreement,
        'recommendation': recommendation,
    }


def get_model_info() -> dict:
    """Return performance metrics for both models."""
    return {
        'models': [
            {
                'name':                    'TF-IDF + Logistic Regression',
                'speed':                   'instant (< 1 ms per tweet)',
                'macro_f1_validation':     0.741,
                'weighted_f1_validation':  0.804,
                'accuracy_validation':     0.799,
                'best_use': 'Exploratory analysis, offline use, high-throughput pipelines.',
            },
            {
                'name':                    'Fine-tuned FinBERT (ProsusAI/finbert)',
                'speed':                   '~1 s per tweet (one-time ~10 s load)',
                'macro_f1_validation':     0.822,
                'weighted_f1_validation':  0.865,
                'accuracy_validation':     0.862,
                'per_class_f1':            {'Bearish': 0.758, 'Bullish': 0.799, 'Neutral': 0.910},
                'best_use': 'Production use, high-stakes decisions, imbalanced classes.',
            },
        ],
        'recommendation': (
            'Use fine-tuned FinBERT when accuracy matters. '
            'Use TF-IDF + LogReg when speed or resource constraints apply.'
        ),
    }


print('Tool functions defined: classify_fast, classify_best, compare_models, get_model_info')


In [ ]:
TOOL_SCHEMAS = [
    {
        'type': 'function',
        'function': {
            'name': 'classify_fast',
            'description': (
                'Classify a financial tweet using the fast TF-IDF + Logistic Regression model. '
                'Use for quick or exploratory classification. Macro F1 = 0.741 on validation set.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to classify.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'classify_best',
            'description': (
                'Classify a financial tweet using fine-tuned FinBERT, the most accurate model. '
                'Returns the predicted label plus per-class confidence scores. '
                'Macro F1 = 0.822 on validation set.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to classify.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'compare_models',
            'description': (
                'Run both the fast and the best model on the same tweet and compare their '
                'predictions. Use when the tweet seems ambiguous, the user asks for '
                'verification, or you want to check whether the classifiers agree.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to compare.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_model_info',
            'description': (
                'Return performance metrics and descriptions for both available models. '
                'Use when the user asks about accuracy, reliability, or which model to trust.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {},
                'required': [],
            },
        },
    },
]

print(f'{len(TOOL_SCHEMAS)} tool schemas defined.')


In [ ]:
load_dotenv(PROJECT_ROOT / '.env', override=False)

_api_key  = os.getenv('OPENAI_API_KEY') or os.getenv('DEEPSEEK_API_KEY')
_api_base = os.getenv('OPENAI_API_BASE', 'https://api.deepseek.com/v1')
_model    = os.getenv('OPENAI_MODEL', 'deepseek-chat')

if not _api_key:
    raise RuntimeError('No API key found. Set OPENAI_API_KEY or DEEPSEEK_API_KEY in .env')

client = OpenAI(api_key=_api_key, base_url=_api_base)

SYSTEM_PROMPT = (
    'You are a financial tweet sentiment analysis agent. '
    'You classify tweets about financial markets into one of three categories:\n'
    '  - Bearish (0): negative market outlook — decline expectations, negative earnings\n'
    '  - Bullish (1): positive market outlook — growth expectations, positive earnings\n'
    '  - Neutral (2): no clear directional sentiment — factual news, announcements\n\n'
    'You have two classification tools:\n'
    '  - classify_fast:  TF-IDF + Logistic Regression (macro F1 = 0.741, < 1 ms)\n'
    '  - classify_best:  Fine-tuned FinBERT           (macro F1 = 0.822, ~1 s)\n\n'
    'Decision protocol:\n'
    '1. Quick or exploratory requests          → classify_fast\n'
    '2. User asks for most accurate result     → classify_best\n'
    '3. Ambiguous tweet or verification needed → compare_models\n'
    '4. Fast and best disagree                 → compare_models, explain discrepancy\n'
    '5. User asks about model performance      → get_model_info\n\n'
    'Always explain the sentiment in a financial context and state your confidence.'
)

TOOL_DISPATCH = {
    'classify_fast':  lambda args: classify_fast(args['tweet']),
    'classify_best':  lambda args: classify_best(args['tweet']),
    'compare_models': lambda args: compare_models(args['tweet']),
    'get_model_info': lambda _:    get_model_info(),
}


def _msg_to_dict(msg) -> dict:
    d = {'role': 'assistant', 'content': msg.content or ''}
    if msg.tool_calls:
        d['tool_calls'] = [
            {
                'id':   tc.id,
                'type': 'function',
                'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
            }
            for tc in msg.tool_calls
        ]
    return d


def run_agent(user_message: str, history: list) -> tuple[str, list]:
    """Send one user turn to the agent and return (response_text, updated_history)."""
    history = history + [{'role': 'user', 'content': user_message}]
    while True:
        response = client.chat.completions.create(
            model=_model,
            messages=[{'role': 'system', 'content': SYSTEM_PROMPT}] + history,
            tools=TOOL_SCHEMAS,
            tool_choice='auto',
        )
        msg = response.choices[0].message
        history = history + [_msg_to_dict(msg)]

        if not msg.tool_calls:
            return msg.content, history

        for tc in msg.tool_calls:
            result = TOOL_DISPATCH[tc.function.name](json.loads(tc.function.arguments))
            history = history + [{
                'role':         'tool',
                'tool_call_id': tc.id,
                'content':      json.dumps(result),
            }]


def chat():
    """Start an interactive conversational session with the agent."""
    history = []
    print('=' * 60)
    print('Financial Sentiment Analysis Agent')
    print("Type a tweet or question. Type 'quit' to exit.")
    print('=' * 60)
    print()
    while True:
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nSession ended.')
            break
        if not user_input:
            continue
        if user_input.lower() in {'quit', 'exit', 'q'}:
            print('Goodbye!')
            break
        response, history = run_agent(user_input, history)
        print(f'\nAgent: {response}\n')


print(f'Agent ready. API base: {_api_base}  |  Model: {_model}')


## Interactive Session

Run the cell below to start a live conversation with the agent.
Type any financial tweet or question; the agent will decide which
model to use. Type `quit` to end the session.


In [ ]:
# Uncomment to start an interactive chat session:
# chat()

print('Uncomment chat() above to start an interactive session.')
print('The scripted demo below (next cell) runs automatically.')


## Demo Conversation

A pre-scripted four-turn conversation that demonstrates all agent capabilities:

1. **Turn 1** — simple request → agent routes to `classify_fast`
2. **Turn 2** — user asks for verification → agent uses `classify_best`
3. **Turn 3** — ambiguous tweet → agent calls `compare_models`, resolves disagreement
4. **Turn 4** — user asks which model to trust → agent calls `get_model_info`


In [ ]:
DEMO_INPUTS = [
    (
        "Quickly classify this tweet: "
        "'AAPL earnings beat expectations by $0.15 EPS. Stock up 5% in after-hours trading.'"
    ),
    "Can you double-check that with your most accurate model?",
    (
        "I am unsure about this one — could be bearish or neutral. "
        "'Fed signals possible rate pause but warns inflation remains above target. "
        "Analysts are divided on the path forward.'"
    ),
    "Which of your two models should I rely on for important trading decisions?",
]

print('=' * 60)
print('DEMO: Financial Sentiment Analysis Agent')
print('=' * 60)
print()

history = []
for user_input in DEMO_INPUTS:
    print(f'You: {user_input}')
    print()
    response, history = run_agent(user_input, history)
    print(f'Agent: {response}')
    print()
    print('-' * 60)
    print()
